In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# 1. Load Datasets
# Make sure 'control_group.xlsx' and 'test_group.xlsx' are in your directory
control_df = pd.read_excel('control_group.xlsx')
test_df = pd.read_excel('test_group.xlsx')

# Clean column names
control_df.columns = [col.lower().replace(' ', '_') for col in control_df.columns]
test_df.columns = [col.lower().replace(' ', '_') for col in test_df.columns]

# 2. Data Cleaning & Handling Missing Values
control_df = control_df.dropna()
test_df = test_df.dropna()

# 3. Define Metric of Interest (Purchases / Conversions)
control_purchases = control_df['purchase']
test_purchases = test_df['purchase']

print(f"Control Group Mean Purchases: {control_purchases.mean():.4f}")
print(f"Test Group Mean Purchases: {test_purchases.mean():.4f}")

# 4. Assumption Checks
# Normality Test (Shapiro-Wilk)
control_shapiro = stats.shapiro(control_purchases)
test_shapiro = stats.shapiro(test_purchases)

print(f"Control Normality P-value: {control_shapiro.pvalue:.4f}")
print(f"Test Normality P-value: {test_shapiro.pvalue:.4f}")

# Variance Homogeneity (Levene's Test)
levene_test = stats.levene(control_purchases, test_purchases)
print(f"Levene P-value: {levene_test.pvalue:.4f}")

# 5. Hypothesis Testing
# H0: µ_control = µ_test (No significant difference)
# H1: µ_control != µ_test (Significant difference)

if control_shapiro.pvalue > 0.05 and test_shapiro.pvalue > 0.05:
  if levene_test.pvalue > 0.05:
    t_stat, p_val = stats.ttest_ind(
        control_purchases, test_purchases, equal_var=True
    )
  else:
    t_stat, p_val = stats.ttest_ind(
        control_purchases, test_purchases, equal_var=False
    )
  test_used = "Independent Two-Sample T-Test"
else:
  u_stat, p_val = stats.mannwhitneyu(control_purchases, test_purchases)
  test_used = "Mann-Whitney U Test"

print(f"Selected Statistical Test: {test_used}")
print(f"P-value: {p_val:.4f}")

# 6. Final Decision
if p_val < 0.05:
  print(
      "Result: Reject H0. There is a statistically significant difference"
      " between strategies."
  )
else:
  print(
      "Result: Fail to reject H0. No statistically significant difference"
      " found."
  )